# 60 Summary over transects

<!-- navigation -->
[<kbd>← 52 Timflow model</kbd>](52%20Timflow%20model.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md)

**Optional**: after notebook 52 for two or more transects

Combines the model summaries and response factors of all transects into one table and figures.

| | |
|---|---|
| **Input** | `model_summary.xlsx` and `responsefactors.xlsx` in the 'results_final' folder of each transect (notebook 52) |
| **Output** | `summary_transects.xlsx`, `responsefactors_transects.xlsx` and figures in the project folder 'results' |

In [ ]:
from pathlib import Path

import timflow_levee_toolkit as tlt

import matplotlib.pyplot as plt
import pandas as pd
print(f"timflow_levee_toolkit {tlt.__version__}")

In [ ]:
## UPDATE FOR YOUR PROJECT: the project folder (with project_settings.toml and the data)
project_dir = Path.cwd().parent / "examples" / "papendrecht"

project = tlt.load_project(project_dir)
units = project.units
project

In [ ]:
transects = project.transects()  # or a selection, e.g. ["Papendrecht West", "Papendrecht East"]

## 1. Collect the results

In [ ]:
summaries, rf_tables = [], []
for transect in transects:
    folder = project.transect_dir(transect, "final")
    fn_summary, fn_rf = folder / "model_summary.xlsx", folder / "responsefactors.xlsx"
    if not (fn_summary.exists() and fn_rf.exists()):
        print(f"{transect}: no results yet (run notebook 52 for this transect)")
        continue
    summaries.append(pd.read_excel(fn_summary, index_col=0))
    rf = pd.read_excel(fn_rf, index_col=0)
    rf.insert(0, "transect", transect)
    rf_tables.append(rf)

summary = pd.concat(summaries)
responsefactors = pd.concat(rf_tables)

project.results_dir.mkdir(parents=True, exist_ok=True)
tlt.export_to_excel(summary, project.results_dir / "summary_transects.xlsx")
rf_columns = [c for c in responsefactors.columns if c.startswith("rf_")]
lag_columns = [c for c in responsefactors.columns if c.startswith("lag_")]
tlt.export_to_excel(
    responsefactors,
    project.results_dir / "responsefactors_transects.xlsx",
    number_formats={**{c: "0.0%" for c in rf_columns}, **{c: "0.0" for c in lag_columns}},
)

summary.T.style.format(precision=3)

## 2. Leakage lengths per transect
Fitted with the exponential decay function on the steady and design hydrograph (MHW) response factors of the deepest layer.

In [ ]:
columns = {
    "foreland, steady": "leakage_length_foreland_steady_m",
    "hinterland, steady": "leakage_length_hinterland_steady_m",
    "foreland, MHW": "leakage_length_foreland_mhw_m",
    "hinterland, MHW": "leakage_length_hinterland_mhw_m",
}
f, ax = plt.subplots(figsize=(10, 5), layout="constrained")
summary[list(columns.values())].rename(columns={v: k for k, v in columns.items()}).plot.bar(ax=ax, rot=0)
ax.set_ylabel(f"leakage length [{units['x']}]")
ax.legend(loc=(0, 1), frameon=False, ncol=4, fontsize="x-small")
ax.grid(True, axis="y")
tlt.finalize_and_save(f, project.results_dir, project.name, tag="leakage_lengths", file_label="")

## 3. Response factors along the cross-section
Modelled response factors in the deepest layer, per transect.

In [ ]:
methods = [c for c in ("rf_steady_mod", "rf_tidal_mod", "rf_mhw_mod") if c in responsefactors.columns]
f, axes = plt.subplots(1, len(methods), figsize=(5 * len(methods), 4), sharey=True, layout="constrained", squeeze=False)
for ax, method in zip(axes[0], methods):
    for transect, group in responsefactors.groupby("transect"):
        deepest = group[group["modellayer"] == group["modellayer"].max()].sort_values("x_crosssection")
        ax.plot(deepest["x_crosssection"], deepest[method], "o-", label=transect)
    ax.set_title(method, loc="right")
    ax.set_xlabel(f"x [{units['x']}]")
    ax.grid(True)
axes[0, 0].set_ylabel("response factor [-]")
axes[0, 0].legend(loc=(0, 1), frameon=False, ncol=4, fontsize="x-small")
tlt.finalize_and_save(f, project.results_dir, project.name, tag="responsefactors_transects", file_label="")

---

<!-- navigation -->
[<kbd>← 52 Timflow model</kbd>](52%20Timflow%20model.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md)